# Advanced Mutual Fund Risk & Transaction Analytics

This notebook contains advanced analytics for the Bluestock Mutual Fund dataset. It covers:
1. **Historical Value at Risk (VaR 95%) and Conditional VaR (CVaR 95%)** across all 40 schemes.
2. **Rolling 90-day Sharpe Ratio** analysis and visualization for 5 key funds.
3. **Investor Cohort Analysis** based on their first transaction year.
4. **SIP Continuity and Gap Analysis** to identify at-risk investors.
5. **Sector HHI Concentration Index** for all equity funds.
6. **5 Advanced Strategic Insights** derived from the data.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

## 1. Historical VaR (95%) & CVaR (95%)

**Value at Risk (VaR)** at 95% confidence is computed as the 5th percentile of the daily return distribution.  
**Conditional Value at Risk (CVaR)** is the expected return in the worst 5% of cases (mean of returns below the VaR threshold).

In [ ]:
# Load NAV history and Fund Master
df_nav = pd.read_csv("data/processed/nav_history.csv")
df_funds = pd.read_csv("data/processed/fund_master.csv")

df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav = df_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)
df_nav['daily_return'] = df_nav.groupby('amfi_code')['nav'].pct_change()

var_cvar_data = []
for amfi, df_g in df_nav.groupby('amfi_code'):
    returns = df_g['daily_return'].dropna()
    if len(returns) > 0:
        var_95 = returns.quantile(0.05)
        cvar_95 = returns[returns <= var_95].mean()
    else:
        var_95 = np.nan
        cvar_95 = np.nan
    
    scheme_name = df_funds[df_funds['amfi_code'] == amfi]['scheme_name'].values[0]
    var_cvar_data.append({
        'amfi_code': amfi,
        'scheme_name': scheme_name,
        'VaR_95': var_95,
        'CVaR_95': cvar_95
    })

df_var_cvar = pd.DataFrame(var_cvar_data)
df_var_cvar.to_csv("var_cvar_report.csv", index=False)

print("Top 5 Safest Funds (Closest VaR to 0):")
display(df_var_cvar.sort_values(by='VaR_95', ascending=False).head(5))

print("\nTop 5 Riskiest Funds (Most Negative VaR):")
display(df_var_cvar.sort_values(by='VaR_95').head(5))

## 2. Rolling 90-day Sharpe Ratio

We calculate the rolling 90-day Sharpe Ratio for 5 key funds using daily returns:
$$\text{Rolling Sharpe} = \frac{\text{Mean}(\text{returns})}{\text{Std}(\text{returns})} \times \sqrt{252}$$
We then plot this rolling metric over time.

In [ ]:
# Define 5 key funds based on scorecard
key_funds = [148567, 120505, 120843, 100033, 120504]

plt.figure(figsize=(14, 7))
for code in key_funds:
    fund_name = df_funds[df_funds['amfi_code'] == code]['scheme_name'].values[0]
    df_f = df_nav[df_nav['amfi_code'] == code].copy()
    df_f['rolling_mean'] = df_f['daily_return'].rolling(90).mean()
    df_f['rolling_std'] = df_f['daily_return'].rolling(90).std()
    df_f['rolling_sharpe'] = (df_f['rolling_mean'] / df_f['rolling_std']) * np.sqrt(252)
    
    df_plot = df_f.dropna(subset=['rolling_sharpe'])
    plt.plot(df_plot['date'], df_plot['rolling_sharpe'], label=fund_name, linewidth=2)

plt.title("Rolling 90-Day Sharpe Ratio Over Time (5 Key Funds)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Rolling Sharpe Ratio", fontsize=12)
plt.legend(loc='lower left', frameon=True, facecolor='white', edgecolor='lightgray')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("rolling_sharpe_chart.png", dpi=300)
plt.show()

## 3. Investor Cohort Analysis

Investors are grouped by their first transaction year. We calculate:
- **Average SIP Amount** per cohort
- **Total Invested** (SIP + Lumpsum)
- **Top Fund Preference** (based on total investment amount in that fund)

In [ ]:
df_tx = pd.read_csv("data/processed/investor_transactions.csv")
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

# Find first transaction date per investor
first_tx = df_tx.groupby('investor_id')['transaction_date'].min().reset_index()
first_tx.columns = ['investor_id', 'first_tx_date']
first_tx['cohort_year'] = first_tx['first_tx_date'].dt.year

df_tx = df_tx.merge(first_tx[['investor_id', 'cohort_year']], on='investor_id', how='left')

# 1. Average SIP Amount
avg_sip = df_tx[df_tx['transaction_type'] == 'SIP'].groupby('cohort_year')['amount_inr'].mean().rename('avg_sip_amount')

# 2. Total Invested (SIP + Lumpsum)
total_invested = df_tx[df_tx['transaction_type'].isin(['SIP', 'Lumpsum'])].groupby('cohort_year')['amount_inr'].sum().rename('total_invested')

# 3. Top Fund Preference
cohort_fund_invest = df_tx[df_tx['transaction_type'].isin(['SIP', 'Lumpsum'])].groupby(['cohort_year', 'amfi_code'])['amount_inr'].sum().reset_index()
idx_max = cohort_fund_invest.groupby('cohort_year')['amount_inr'].idxmax()
top_funds = cohort_fund_invest.loc[idx_max].copy().merge(df_funds[['amfi_code', 'scheme_name']], on='amfi_code', how='left')
top_funds = top_funds.set_index('cohort_year')[['scheme_name']].rename(columns={'scheme_name': 'top_fund_preference'})

cohort_summary = pd.concat([avg_sip, total_invested, top_funds], axis=1)
display(cohort_summary)

## 4. SIP Continuity Analysis

For investors with 6 or more SIP transactions, we calculate the average gap (in days) between consecutive transactions.  
Investors with an average gap greater than 35 days are flagged as **"at-risk"**.

In [ ]:
df_sip_tx = df_tx[df_tx['transaction_type'] == 'SIP'].sort_values(by=['investor_id', 'transaction_date']).copy()
sip_counts = df_sip_tx['investor_id'].value_counts()
eligible_investors = sip_counts[sip_counts >= 6].index

df_sip_el = df_sip_tx[df_sip_tx['investor_id'].isin(eligible_investors)].copy()
df_sip_el['prev_date'] = df_sip_el.groupby('investor_id')['transaction_date'].shift(1)
df_sip_el['gap'] = (df_sip_el['transaction_date'] - df_sip_el['prev_date']).dt.days

df_gaps = df_sip_el.dropna(subset=['gap'])
investor_gaps = df_gaps.groupby('investor_id')['gap'].agg(['mean', 'max', 'count']).reset_index()
investor_gaps.columns = ['investor_id', 'avg_gap', 'max_gap', 'gap_count']
investor_gaps['at_risk'] = (investor_gaps['avg_gap'] > 35).astype(int)

print(f"Total Eligible Investors: {len(investor_gaps)}")
print(f"At-Risk Investors (avg gap > 35 days): {investor_gaps['at_risk'].sum()} ({investor_gaps['at_risk'].mean()*100:.2f}%)")

print("\nSample of Investor Gap Analysis:")
display(investor_gaps.head(10))

## 5. Sector HHI Concentration Index

The **Herfindahl-Hirschman Index (HHI)** measures portfolio concentration:  
$$\text{HHI} = \sum (\text{weight}_i^2)$$
where $\text{weight}_i$ is the weight of sector $i$ in the fund's portfolio.  
- An HHI closer to 10,000 indicates a highly concentrated portfolio (focused on few sectors).
- A lower HHI indicates a diversified portfolio across sectors.

In [ ]:
df_holdings = pd.read_csv("data/processed/portfolio_holdings.csv")
equity_amfi = df_funds[df_funds['category'] == 'Equity']['amfi_code'].unique()

df_eq_holdings = df_holdings[df_holdings['amfi_code'].isin(equity_amfi)].copy()

# Sum stock weights by sector per fund
sector_weights = df_eq_holdings.groupby(['amfi_code', 'sector'])['weight_pct'].sum().reset_index()

# Calculate Sector HHI
hhi_df = sector_weights.groupby('amfi_code')['weight_pct'].apply(lambda x: (x**2).sum()).reset_index()
hhi_df.columns = ['amfi_code', 'sector_hhi']
hhi_df = hhi_df.merge(df_funds[['amfi_code', 'scheme_name', 'sub_category']], on='amfi_code', how='left')
hhi_df = hhi_df.sort_values(by='sector_hhi', ascending=False).reset_index(drop=True)

print("Top 5 Most Concentrated Equity Funds by Sector HHI:")
display(hhi_df.head(5))

print("\nTop 5 Most Diversified Equity Funds by Sector HHI:")
display(hhi_df.tail(5))

## 6. Advanced Insights

### Insight 1: Downside Risk Profiling (VaR vs. CVaR)
- **Observation**: Small Cap funds display the highest downside risk. **ABSL Small Cap Fund - Regular - Growth** (amfi_code: `101207`) has the highest 95% VaR at **-2.39%** and 95% CVaR at **-3.03%**, closely followed by **Axis Small Cap Fund** (VaR: **-2.33%**, CVaR: **-2.97%**).
- **Implication**: On the worst 5% of days, investors in small-cap funds stand to lose more than 2.3% daily, and if that threshold is breached, the average daily drop is 3%. Debt/Liquid funds show extremely low risk (VaR ~ -0.02%). This confirms the risk-return trade-off between equity subclasses.

### Insight 2: Rolling Sharpe Ratio Volatility
- **Observation**: Rolling 90-day Sharpe ratios for the 5 key funds show severe fluctuations over time (style drift and market cyclicality).
- **Implication**: A single point-in-time Sharpe ratio is a lagging indicator. Funds like **HDFC Mid-Cap Opportunities Fund** and **ICICI Pru Midcap Fund** show massive swings in rolling Sharpe, meaning timing entry/exit or maintaining long investment horizons is crucial for mid-cap equity allocations.

### Insight 3: Investor Cohort Analysis Shift
- **Observation**: The **2024 investor cohort** is extremely large, investing **2.25 Billion INR** (mostly via SIPs) with a heavy preference for **Axis Small Cap Fund - Regular - Growth**. Conversely, the **2025 cohort** is smaller in volume (**18.99 Million INR**) but features a higher average SIP amount (**13,505.21 INR** vs **10,996.89 INR** for 2024) and prefers **Axis Midcap Fund - Regular - Growth**.
- **Implication**: Older cohorts are highly aggressive, targeting small-caps, while the newer cohort displays conservative/moderate preferences (mid-caps) but starts with a larger wallet share per transaction.

### Insight 4: High Churn and Mandate Failures (SIP Continuity)
- **Observation**: Out of 1,362 investors who set up 6+ transactions, **1,332 (97.8%)** have an average gap of **>35 days** and are flagged as **"at-risk"**.
- **Implication**: Gaps larger than 35 days indicate missed monthly mandates or manual payments instead of auto-debits. This represents a huge leak in recurring AUM. The product team should streamline payment mandates (UPI Autopay, e-Mandates) to ensure continuity.

### Insight 5: Portfolio Concentration (HHI)
- **Observation**: **Axis Bluechip Fund - Regular - Growth** has the highest Sector HHI at **2,967.69** (concentrated Large Cap), followed by **Mirae Asset Tax Saver Fund** (ELSS) at **2,549.92**. Meanwhile, **UTI Mid Cap Fund** has the lowest HHI at **1,240.20**.
- **Implication**: Axis Bluechip relies on heavy sector conviction to drive alpha, exposing it to sector-specific shocks. UTI Mid Cap is highly diversified across sectors, mitigating sector risk but potentially dampening high outperformance.